# FOMC Rate Decisions and Market Reaction: Event Study

## Mục tiêu

Xem thị trường có phản ứng với quyết định lãi suất của Fed (FOMC) không. Chia làm 3 câu hỏi:

1. **Q1:** SPY (thị trường chung) có biến động cao hơn quanh các sự kiện FOMC không
2. **Q2:** AAPL, TLT, XLF chỉ đi theo thị trường, hay có mã nào phản ứng mạnh hơn bình thường?
3. **Q3 (extension):** VIX có tăng trước ngày họp và giảm sau đó không?



## 0. Import thư viện

In [1]:
!pip install arch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import statsmodels.api as sm
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model

%matplotlib inline

# Cố định seed để kết quả random (permutation, bootstrap) có thể tái tạo lại được.
RNG = np.random.default_rng(42)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 15.9 MB/s eta 0:00:00


## 1. Đọc dữ liệu và tính return

Hàm dùng chung cho 4 mã (SPY, AAPL, TLT, XLF), để xử lý giống nhau — kể cả bỏ dòng NaN đầu tiên (ngày đầu không có return vì chưa có ngày trước đó để so sánh).

In [2]:
raw_preview = pd.read_csv("spy_2013_2026.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
raw_preview.head()

,Date,Close,High,Low,Open,Volume
0,2013-01-02,115.992897,116.064367,114.936682,115.238461,192059000
1,2013-01-03,115.730820,116.239072,115.421103,115.937305,144761800
2,2013-01-04,116.239075,116.429674,115.683175,115.921421,116817700
3,2013-01-07,115.921440,116.032620,115.492595,115.826147,110002500
4,2013-01-08,115.587875,115.873768,115.135207,115.714942,121265100


In [3]:
def load_price_series(path, name):
    """Đọc 1 file CSV giá (định dạng Yahoo Finance) và trả về DataFrame
    gồm 2 cột: Date và {name}_ret (lợi suất đơn giản mỗi ngày, dạng thập
    phân — ví dụ 0.01 nghĩa là +1%, không phải đã nhân 100)."""

    # skiprows=[1, 2]: bỏ 2 dòng metadata thừa mà Yahoo Finance hay chèn vào.
    raw = pd.read_csv(path, skiprows=[1, 2]).rename(columns={"Price": "Date"})

    # Chuyển cột Date từ string sang datetime.
    raw["Date"] = pd.to_datetime(raw["Date"])

    # Sắp xếp theo thời gian tăng dần.
    raw = raw.sort_values("Date").reset_index(drop=True)

    # Lợi suất đơn giản mỗi ngày: return_t = Close_t / Close_(t-1) - 1
    raw[f"{name}_ret"] = raw["Close"].pct_change()

    # Dòng đầu tiên luôn là NaN (không có ngày trước đó để so sánh).
    # Loại bỏ ngay tại đây để cả 4 mã được xử lý đồng nhất, thay vì chỉ dropna()
    # riêng cho SPY như bản trước.
    raw = raw.dropna(subset=[f"{name}_ret"]).reset_index(drop=True)

    return raw[["Date", f"{name}_ret"]]


# Đọc dữ liệu SPY, dùng làm đại diện cho thị trường chung.
spy = load_price_series("spy_2013_2026.csv", "spy").rename(columns={"spy_ret": "ret"})

# Đọc dữ liệu 3 mã còn lại.
aapl = load_price_series("aapl_2013_2026.csv", "aapl")
tlt = load_price_series("tlt_2013_2026.csv", "tlt")
xlf = load_price_series("xlf_2013_2026.csv", "xlf")

# Danh sách các ngày giao dịch của SPY, dùng để căn ngày sự kiện FOMC
# vào đúng ngày giao dịch gần nhất.
trading_dates = sorted(spy["Date"].tolist())
trading_dates_set = set(trading_dates)

print(f"SPY: {len(spy)} ngày giao dịch "
      f"({spy['Date'].min().date()} đến {spy['Date'].max().date()})")


SPY: 3430 ngày giao dịch (2013-01-03 đến 2026-08-24)


In [4]:
aapl.head()

,Date,aapl_ret
0,2013-01-03,-0.012622
1,2013-01-04,-0.027855
2,2013-01-07,-0.005882
3,2013-01-08,0.002691
4,2013-01-09,-0.015629


## 2. Kiểm tra chất lượng dữ liệu

Kiểm tra: có thiếu dữ liệu không, có ngày trùng không, dữ liệu đã sắp xếp đúng thứ tự chưa.

In [5]:
assets = {"SPY": spy, "AAPL": aapl, "TLT": tlt, "XLF": xlf}

for name, df in assets.items():
    print(f"\n{name}")
    print("-" * 40)
    print(f"Số dòng dữ liệu: {len(df)}")
    print(f"Khoảng thời gian: {df['Date'].min().date()} đến {df['Date'].max().date()}")
    print("Số giá trị bị thiếu (NaN):")
    print(df.isna().sum())
    print(f"Số ngày bị trùng: {df['Date'].duplicated().sum()}")
    # Kiểm tra thứ tự thời gian: Date1​<Date2​<⋯<DateT​
    print(f"Ngày đã được sắp xếp tăng dần: {df['Date'].is_monotonic_increasing}")



SPY
----------------------------------------
Số dòng dữ liệu: 3430
Khoảng thời gian: 2013-01-03 đến 2026-08-24
Số giá trị bị thiếu (NaN):
Date    0
ret     0
dtype: int64
Số ngày bị trùng: 0
Ngày đã được sắp xếp tăng dần: True

AAPL
----------------------------------------
Số dòng dữ liệu: 3430
Khoảng thời gian: 2013-01-03 đến 2026-08-24
Số giá trị bị thiếu (NaN):
Date        0
aapl_ret    0
dtype: int64
Số ngày bị trùng: 0
Ngày đã được sắp xếp tăng dần: True

TLT
----------------------------------------
Số dòng dữ liệu: 3430
Khoảng thời gian: 2013-01-03 đến 2026-08-24
Số giá trị bị thiếu (NaN):
Date       0
tlt_ret    0
dtype: int64
Số ngày bị trùng: 0
Ngày đã được sắp xếp tăng dần: True

XLF
----------------------------------------
Số dòng dữ liệu: 3430
Khoảng thời gian: 2013-01-03 đến 2026-08-24
Số giá trị bị thiếu (NaN):
Date       0
xlf_ret    0
dtype: int64
Số ngày bị trùng: 0
Ngày đã được sắp xếp tăng dần: True


In [6]:
# Kiểm tra xem 3 mã còn lại có cùng bộ ngày giao dịch với SPY không.
# Nếu lệch nhiều, các bước merge phía sau có thể làm mất dữ liệu.
spy_dates = set(spy["Date"])

for name, df in {"AAPL": aapl, "TLT": tlt, "XLF": xlf}.items():
    asset_dates = set(df["Date"])
    missing_from_asset = spy_dates - asset_dates
    extra_in_asset = asset_dates - spy_dates

    print(f"\n{name}")
    print(f"Ngày có ở SPY nhưng thiếu ở {name}: {len(missing_from_asset)}")
    print(f"Ngày có ở {name} nhưng không có ở SPY: {len(extra_in_asset)}")



AAPL
Ngày có ở SPY nhưng thiếu ở AAPL: 0
Ngày có ở AAPL nhưng không có ở SPY: 0

TLT
Ngày có ở SPY nhưng thiếu ở TLT: 0
Ngày có ở TLT nhưng không có ở SPY: 0

XLF
Ngày có ở SPY nhưng thiếu ở XLF: 0
Ngày có ở XLF nhưng không có ở SPY: 0


## 3. Gán ngày công bố FOMC vào ngày giao dịch

Mỗi thông báo có ngày công bố và giờ công bố (trong giờ hay sau giờ). Cần tìm đúng ngày mà thị trường có thể phản ứng:

- Công bố **sau giờ**: ngày phản ứng là ngày giao dịch kế tiếp.
- Công bố **trong giờ**: ngày phản ứng là chính ngày đó (hoặc ngày giao dịch gần nhất sau đó).

In [7]:
fomc_events = pd.read_csv("fomc_events.csv")
fomc_events["announcement_date"] = pd.to_datetime(fomc_events["announcement_date"])

# Chỉ giữ các sự kiện nằm trong khoảng thời gian mà mình có dữ liệu giá SPY.
fomc_events = fomc_events[
    (fomc_events["announcement_date"] >= spy["Date"].min()) &
    (fomc_events["announcement_date"] <= spy["Date"].max())
].reset_index(drop=True)


def map_to_event_day(ann_date, timing, trading_dates_sorted):
    """Tìm ngày giao dịch mà thị trường phản ứng với thông báo FOMC."""
    if timing == "after_hours":
        candidates = [d for d in trading_dates_sorted if d > ann_date]
    else:
        candidates = [d for d in trading_dates_sorted if d >= ann_date]
    return candidates[0] if candidates else None


fomc_events["event_trading_day"] = fomc_events.apply(
    lambda r: map_to_event_day(r["announcement_date"], r["timing"], trading_dates),
    axis=1,
)

# Cửa sổ sự kiện là 3 ngày: [-1, 0, +1] quanh ngày phản ứng.
WINDOW = 1
event_windows = {}

for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in trading_dates_set:
        continue
    idx = trading_dates.index(ed)
    lo, hi = idx - WINDOW, idx + WINDOW
    if lo >= 0 and hi < len(trading_dates):
        event_windows[ed] = list(range(lo, hi + 1))

print(f"Số sự kiện trong mẫu: {len(fomc_events)} | Số event window đã tạo: {len(event_windows)}")
fomc_events[["announcement_date", "direction", "scheduled", "event_trading_day"]]


Số sự kiện trong mẫu: 31 | Số event window đã tạo: 31


,announcement_date,direction,scheduled,event_trading_day
0,2015-12-16,hike,True,2015-12-16
1,2016-12-14,hike,True,2016-12-14
2,2017-03-15,hike,True,2017-03-15
3,2017-06-14,hike,True,2017-06-14
4,2017-12-13,hike,True,2017-12-13
5,2018-03-21,hike,True,2018-03-21
6,2018-06-13,hike,True,2018-06-13
7,2018-09-26,hike,True,2018-09-26
8,2018-12-19,hike,True,2018-12-19
9,2019-07-31,cut,True,2019-07-31


In [8]:
# Kiểm tra xem có 2 thông báo nào bị gán vào cùng 1 ngày giao dịch không.
duplicate_event_days = fomc_events[
    fomc_events["event_trading_day"].duplicated(keep=False)
].sort_values("event_trading_day")

print("Các ngày sự kiện bị trùng (nếu có):")
print(duplicate_event_days[["announcement_date", "timing", "event_trading_day"]])


Các ngày sự kiện bị trùng (nếu có):
Empty DataFrame
Columns: [announcement_date, timing, event_trading_day]
Index: []


In [9]:
# Kiểm tra các sự kiện không gán được vào ngày giao dịch nào (thiếu dữ liệu giá).
unmapped_events = fomc_events[fomc_events["event_trading_day"].isna()]

print(f"Số sự kiện không gán được: {len(unmapped_events)}")
if len(unmapped_events) > 0:
    display(unmapped_events)


Số sự kiện không gán được: 0


### 3b. Kiểm tra event window có bị trùng nhau không

Mỗi event window dài 3 ngày. Nếu 2 sự kiện FOMC quá gần nhau, một ngày có thể bị tính vào 2 window cùng lúc, làm sai kết quả Q1. Nên kiểm tra rõ, không nên chỉ giả định là không có.

In [10]:
all_window_rows = []
for ed, idx_list in event_windows.items():
    for idx in idx_list:
        all_window_rows.append({
            "event_date": ed,
            "index": idx,
            "date": trading_dates[idx],
        })

window_check = pd.DataFrame(all_window_rows)

# Nếu một "date" xuất hiện nhiều hơn 1 lần trong bảng này, nghĩa là ngày đó
# thuộc về nhiều hơn 1 event window -> có overlap.
overlapping_dates = window_check[
    window_check.duplicated("date", keep=False)
].sort_values("date")

print(f"Số ngày giao dịch bị chồng lấn giữa các event window: {len(overlapping_dates)}")

if len(overlapping_dates) > 0:
    display(overlapping_dates)
else:
    print("Không có overlap - mỗi ngày giao dịch chỉ thuộc về đúng 1 event window.")


Số ngày giao dịch bị chồng lấn giữa các event window: 0
Không có overlap - mỗi ngày giao dịch chỉ thuộc về đúng 1 event window.


## 4. Q1 - SPY có biến động cao hơn quanh các sự kiện FOMC không?

 **GARCH(1,1) conditional volatility**: conditional volatility là độ lệch chuẩn có điều kiện của return tại ngày $t$, ước lượng dựa trên thông tin quá khứ:

$$
\sigma_t = \sqrt{Var(R_t \mid \mathcal{F}_{t-1})}
$$


**Lưu ý về cách ước lượng — tránh look-ahead:** Sử dụng cơ chế walk-forward, expanding-window estimation. Mô hình được refit định kỳ chỉ bằng dữ liệu lịch sử tính đến thời điểm đó, ngăn chặn việc tham số bị ảnh hưởng bởi dữ liệu tương lai.

**Hạn chế:** Nhóm "non-event" chọn ngẫu nhiên và chưa được chuẩn hóa theo giai đoạn/chế độ biến động (volatility regime). Chênh lệch kết quả có thể do chu kỳ thị trường (ví dụ: khủng hoảng 2020) chứ không thuần túy là hiệu ứng FOMC.

In [11]:
def walk_forward_garch_conditional_vol(ret_pct, min_hist=750, refit_every=250):
    """Tính conditional volatility kiểu walk-forward với EXPANDING WINDOW
    (không phải rolling fixed window) — không dùng dữ liệu tương lai để
    CHỌN tham số tại bất kỳ thời điểm nào.

    - Từ ngày `min_hist` trở đi (mặc định 750 ngày ~ 3 năm, GARCH cần đủ
      lịch sử mới ước lượng ổn định), GARCH(1,1) được refit định kỳ mỗi
      `refit_every` ngày (mặc định 250 ngày ~ 1 năm). Mỗi lần refit dùng
      TOÀN BỘ dữ liệu từ đầu chuỗi đến thời điểm đó (expanding window,
      cửa sổ ước lượng dài dần theo thời gian) — chứ không phải một cửa
      sổ có độ dài cố định trượt qua từng năm.
    - Giữa 2 lần refit, (omega, alpha, beta) giữ cố định; sigma_t vẫn cập
      nhật mỗi ngày theo đúng công thức GARCH, dùng return thực tế đã biết
      tại từng thời điểm.
    - Các ngày trước `min_hist` không có đủ dữ liệu -> trả về NaN.
    """
    n = len(ret_pct)
    cond_vol = np.full(n, np.nan)

    sigma2 = eps2 = omega = alpha = beta = mu = None

    for t in range(min_hist, n):
        # Đến lúc refit: dùng dữ liệu từ đầu đến t-1 (không gồm ngày t).
        if omega is None or (t - min_hist) % refit_every == 0:
            fit = arch_model(ret_pct[:t], vol="Garch", p=1, q=1, dist="t").fit(disp="off")
            omega = fit.params["omega"]
            alpha = fit.params["alpha[1]"]
            beta = fit.params["beta[1]"]
            mu = fit.params["mu"]
            sigma2 = fit.conditional_volatility[-1] ** 2
            eps2 = fit.resid[-1] ** 2

        # Cập nhật 1 bước theo công thức GARCH, tham số giữ cố định:
        # sigma_t^2 = omega + alpha * eps_(t-1)^2 + beta * sigma_(t-1)^2
        # -> sigma_t chỉ dùng thông tin đến t-1 (filtered, không phải forecast xa).
        sigma2 = omega + alpha * eps2 + beta * sigma2
        cond_vol[t] = np.sqrt(sigma2)

        eps_t = ret_pct[t] - mu
        eps2 = eps_t ** 2

    return cond_vol


# Nhân return với 100 (thực hành chuẩn của thư viện arch để ổn định số học).
raw_cond_vol = walk_forward_garch_conditional_vol(spy["ret"].values * 100, min_hist=750, refit_every=250)
spy["cond_vol"] = raw_cond_vol / 100  # đưa về cùng scale với ret gốc

n_missing = spy["cond_vol"].isna().sum()
print(f"Số ngày không có cond_vol (chưa đủ lịch sử, ~{n_missing/252:.1f} năm đầu): {n_missing}")
print(spy[["Date", "ret", "cond_vol"]].tail())


Số ngày không có cond_vol (chưa đủ lịch sử, ~3.0 năm đầu): 750
           Date       ret  cond_vol
3425 2026-08-18 -0.006756  0.006970
3426 2026-08-19  0.002098  0.007273
3427 2026-08-20 -0.008400  0.006743
3428 2026-08-21  0.004091  0.007455
3429 2026-08-24 -0.002938  0.007008


**Cách làm:** mỗi sự kiện lấy 1 cửa sổ 3 ngày `[-1, 0, +1]`, tính trung bình `cond_vol` mỗi ngày. So sánh số này giữa nhóm "có sự kiện" và nhóm "không có sự kiện". Vẫn dùng 4 phép kiểm định khác nhau để kết quả không phụ thuộc vào 1 phương pháp duy nhất.

**Lưu ý:** vì walk-forward GARCH cần tối thiểu ~3 năm dữ liệu lịch sử mới bắt đầu tính `cond_vol`, một vài sự kiện FOMC sớm nhất (2015-2016) có thể không có đủ dữ liệu và sẽ bị loại khỏi mẫu — cell dưới sẽ in rõ còn lại bao nhiêu sự kiện.

In [12]:
# --- Tính chỉ số cho từng event window ---
event_level_rows = []
for ed, idx_list in event_windows.items():
    w = spy.iloc[idx_list]
    event_level_rows.append({
        "event_date": ed,
        "avg_ret": w["ret"].mean(),
        "avg_cond_vol": w["cond_vol"].mean(),  # NaN nếu window rơi vào vùng chưa đủ lịch sử
    })

event_level_df = (
    pd.DataFrame(event_level_rows)
    .merge(
        fomc_events[["event_trading_day", "direction", "scheduled"]],
        left_on="event_date", right_on="event_trading_day", how="left",
    )
    .drop(columns="event_trading_day")
    .sort_values("event_date")
    .reset_index(drop=True)
)

n_before = len(event_level_df)
event_level_df = event_level_df.dropna(subset=["avg_cond_vol"]).reset_index(drop=True)
n_after = len(event_level_df)
print(f"Số sự kiện có đủ lịch sử để tính walk-forward GARCH: {n_after}/{n_before}")

# --- Xây dựng các non-event window (nhóm so sánh) ---
# Loại bỏ luôn cả vùng đệm (buffer) 2 ngày quanh mỗi event window, để tránh
# lấy nhầm những ngày "gần sự kiện" làm mẫu "bình thường".
BUFFER = 2
excluded_idx = set()
for ed in event_windows:
    idx = trading_dates.index(ed)
    for offset in range(-WINDOW - BUFFER, WINDOW + BUFFER + 1):
        i = idx + offset
        if 0 <= i < len(trading_dates):
            excluded_idx.add(i)

eligible_idx = sorted(i for i in range(len(spy)) if i not in excluded_idx)

# Gom các ngày "hợp lệ" liên tiếp thành từng cụm 3 ngày.
non_event_stats = []
i = 0
while i + 2 < len(eligible_idx):
    if eligible_idx[i + 2] - eligible_idx[i] == 2:
        r = spy.iloc[eligible_idx[i]: eligible_idx[i] + 3]
        non_event_stats.append({
            "avg_ret": r["ret"].mean(),
            "avg_cond_vol": r["cond_vol"].mean(),
        })
        i += 3
    else:
        i += 1

non_event_df = pd.DataFrame(non_event_stats).dropna(subset=["avg_cond_vol"]).reset_index(drop=True)

print(f"Số event window: {len(event_level_df)} | Số non-event window: {len(non_event_df)}")


Số sự kiện có đủ lịch sử để tính walk-forward GARCH: 30/31
Số event window: 30 | Số non-event window: 814


### 4b. Sample-size sensitivity check

Nhóm event thường nhỏ hơn nhiều so với nhóm non-event (xem số thực tế được in ra ở cell trên — có thể không phải 31, vì walk-forward GARCH loại bớt sự kiện thiếu lịch sử). Đây **không phải bootstrap CI** hay một phép kiểm định thống kê chính thức — nó chỉ trả lời câu hỏi hẹp hơn: *"nếu subsample nhóm non-event xuống cùng cỡ với nhóm event, chênh lệch trung bình có còn dương không?"*. Lặp lại nhiều lần: mỗi lần chọn ngẫu nhiên đúng bằng số event non-event window (theo `avg_cond_vol`) rồi so với nhóm event.

**Lưu ý:** chỉ phía non-event được chọn lại ngẫu nhiên, còn phía event giữ nguyên. Nếu khoảng [2.5, 97.5] không chứa 0, đó là dấu hiệu effect không nhạy cảm với việc pool non-event quá lớn — nhưng không nên trích dẫn như một confidence interval hay p-value chính thức.

In [13]:
event_x = event_level_df["avg_cond_vol"].values
non_event_y = non_event_df["avg_cond_vol"].values

n_event = len(event_x)
n_iter = 10_000

matched_diffs = np.empty(n_iter)

for i in range(n_iter):
    # Mỗi lần lặp, chọn ngẫu nhiên (không lặp lại) đúng n_event non-event
    # window, rồi tính chênh lệch trung bình so với nhóm event (event_x.mean()
    # không đổi qua các vòng lặp — chỉ non-event được resample).
    sampled_non_event = RNG.choice(non_event_y, size=n_event, replace=False)
    matched_diffs[i] = event_x.mean() - sampled_non_event.mean()

low, high = np.percentile(matched_diffs, [2.5, 97.5])

print(f"Subsample non-event xuống n={n_event}, lặp lại {n_iter} lần:")
print(f"  Chênh lệch trung bình = {matched_diffs.mean():+.5f}")
print(f"  Khoảng phần trăm [2.5, 97.5] = [{low:.5f}, {high:.5f}]")
print()
print("Đây KHÔNG phải bootstrap CI đúng nghĩa (chỉ non-event được resample,")
print("event giữ cố định). Chỉ dùng làm chỉ báo: nếu khoảng này không chứa 0,")
print("kết luận ở Q1 có vẻ không nhạy cảm với việc pool non-event quá lớn —")
print("không nên trích dẫn như một kiểm định thống kê độc lập.")


Subsample non-event xuống n=30, lặp lại 10000 lần:
  Chênh lệch trung bình = +0.00290
  Khoảng phần trăm [2.5, 97.5] = [0.00084, 0.00438]

Đây KHÔNG phải bootstrap CI đúng nghĩa (chỉ non-event được resample,
event giữ cố định). Chỉ dùng làm chỉ báo: nếu khoảng này không chứa 0,
kết luận ở Q1 có vẻ không nhạy cảm với việc pool non-event quá lớn —
không nên trích dẫn như một kiểm định thống kê độc lập.


### 4c. Chạy các phép kiểm định cho Q1

Hàm `run_tests` gộp 4 phép kiểm định. Phần permutation test viết rõ hơn: mỗi vòng lặp tạo ra một cách sắp xếp ngẫu nhiên mới bằng `rng.permutation(...)`, dễ đọc và dễ kiểm tra lại hơn.

In [14]:
def run_tests(x, y, label, n_iter=10_000, rng=RNG):
    # 1) Welch's t-test: so sánh trung bình, không giả định 2 nhóm có
    #    phương sai bằng nhau.
    t_stat, p_welch = stats.ttest_ind(x, y, equal_var=False)

    # 2) Mann-Whitney U: phép kiểm định phi tham số, không giả định phân
    #    phối chuẩn.
    u_stat, p_mw = mannwhitneyu(x, y, alternative="two-sided")

    # 3) Permutation test: trộn ngẫu nhiên 2 nhóm lại với nhau nhiều lần,
    #    xem chênh lệch trung bình quan sát được có "bất thường" so với
    #    chênh lệch khi trộn ngẫu nhiên hay không.
    observed = x.mean() - y.mean()
    combined = np.concatenate([x, y])
    n_x = len(x)

    perm_diffs = np.empty(n_iter)
    for k in range(n_iter):
        permuted = rng.permutation(combined)
        perm_diffs[k] = permuted[:n_x].mean() - permuted[n_x:].mean()

    p_perm = np.mean(np.abs(perm_diffs) >= np.abs(observed))

    # 4) Bootstrap 95% CI cho chênh lệch trung bình.
    boot_diffs = np.array([
        rng.choice(x, len(x), replace=True).mean()
        - rng.choice(y, len(y), replace=True).mean()
        for _ in range(n_iter)
    ])
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  Trung bình event={x.mean():.5f}  non-event={y.mean():.5f}  chênh lệch={observed:.5f}")
    print(f"  Welch t-test:  p={p_welch:.4f}   Mann-Whitney: p={p_mw:.4f}")
    print(f"  Permutation:   p={p_perm:.4f}   Bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()


print("=== Kiểm tra hướng của return ===")
run_tests(event_level_df["avg_ret"].values, non_event_df["avg_ret"].values, "Mean return")

print("=== Độ biến động (GARCH conditional volatility) - có volatility tăng không? ===")
run_tests(event_level_df["avg_cond_vol"].values, non_event_df["avg_cond_vol"].values,
           f"Toàn bộ mẫu ({len(event_level_df)} sự kiện)")

# Loại 2 sự kiện cắt lãi suất khẩn cấp (Mar 2020) khỏi nhóm event, xem kết
# quả có còn đứng vững không. Lưu ý: cách này chỉ loại các sự kiện đó (và
# vùng đệm quanh nó) khỏi nhóm event, KHÔNG loại biến động COVID khỏi nhóm
# non-event, nên đây là một robustness check nhỏ, chưa phải kiểm soát COVID
# đầy đủ.
emergency_mask = ~event_level_df["scheduled"]
n_emergency = int(emergency_mask.sum())
n_remaining = len(event_level_df) - n_emergency
run_tests(
    event_level_df.loc[~emergency_mask, "avg_cond_vol"].values,
    non_event_df["avg_cond_vol"].values,
    f"Loại {n_emergency} sự kiện cắt lãi suất khẩn cấp (Mar 2020) khỏi nhóm event (còn {n_remaining} sự kiện)",
)


=== Kiểm tra hướng của return ===
--- Mean return ---
  Trung bình event=0.00005  non-event=0.00087  chênh lệch=-0.00083
  Welch t-test:  p=0.6054   Mann-Whitney: p=0.1669
  Permutation:   p=0.4144   Bootstrap 95% CI: [-0.00382, 0.00226]

=== Độ biến động (GARCH conditional volatility) - có volatility tăng không? ===
--- Toàn bộ mẫu (30 sự kiện) ---
  Trung bình event=0.01220  non-event=0.00930  chênh lệch=0.00290
  Welch t-test:  p=0.1868   Mann-Whitney: p=0.4728
  Permutation:   p=0.0105   Bootstrap 95% CI: [-0.00048, 0.00760]

--- Loại 2 sự kiện cắt lãi suất khẩn cấp (Mar 2020) khỏi nhóm event (còn 28 sự kiện) ---
  Trung bình event=0.00975  non-event=0.00930  chênh lệch=0.00045
  Welch t-test:  p=0.6376   Mann-Whitney: p=0.9135
  Permutation:   p=0.6475   Bootstrap 95% CI: [-0.00128, 0.00238]



**Kết luận Q1:** SPY có volatility cao hơn quanh FOMC trong toàn bộ mẫu, nhưng kết quả giữa các kiểm định chưa đồng nhất. Khi loại 2 lần cắt lãi suất khẩn cấp tháng 3/2020, chênh lệch nhỏ và không còn có ý nghĩa thống kê.

## 5. Q2a - Các mã có đi theo thị trường quanh sự kiện FOMC không?

Trước khi nói một mã phản ứng "bất thường", cần biết nó có đi cùng chiều với thị trường không **trong đúng bối cảnh đang xét (quanh sự kiện FOMC)** — đây chưa phải correlation toàn thời kỳ hay unconditional beta. AAPL, XLF là cổ phiếu nên có thể kỳ vọng đi theo thị trường; TLT (trái phiếu) phản ứng với kỳ vọng lãi suất, không nhất thiết theo tâm lý cổ phiếu.

**Cách làm:** với mỗi cửa sổ 3 ngày, tính return cộng dồn từng mã, so dấu (tăng/giảm) với SPY, và tính tương quan qua 31 sự kiện.

In [15]:
df_all = (
    spy[["Date", "ret"]]
    .rename(columns={"ret": "spy_ret"})
    .merge(aapl, on="Date")
    .merge(tlt, on="Date")
    .merge(xlf, on="Date")
    .dropna()
    .reset_index(drop=True)
)
dates_all = sorted(df_all["Date"].tolist())

comove_rows = []
for _, row in fomc_events.iterrows():
    ed = row["event_trading_day"]
    if ed not in dates_all:
        continue
    idx = dates_all.index(ed)
    if idx - 1 < 0 or idx + 1 >= len(dates_all):
        continue

    w = df_all.iloc[idx - 1: idx + 2]

    # Return cộng dồn đúng cách: (1+R1)(1+R2)(1+R3) - 1
    # (Cộng trực tiếp 3 return lại với nhau chỉ là một cách xấp xỉ.)
    comove_rows.append({
        "event_date": ed.date(),
        "direction": row["direction"],
        "SPY_%": ((1 + w["spy_ret"]).prod() - 1) * 100,
        "AAPL_%": ((1 + w["aapl_ret"]).prod() - 1) * 100,
        "TLT_%": ((1 + w["tlt_ret"]).prod() - 1) * 100,
        "XLF_%": ((1 + w["xlf_ret"]).prod() - 1) * 100,
    })

comove_df = pd.DataFrame(comove_rows)

print("=== % số sự kiện đi cùng hướng với SPY, và hệ số tương quan ===")
for asset in ["AAPL", "TLT", "XLF"]:
    same_dir = (np.sign(comove_df["SPY_%"]) == np.sign(comove_df[f"{asset}_%"])).mean() * 100
    corr = comove_df["SPY_%"].corr(comove_df[f"{asset}_%"])
    print(f"{asset}: {same_dir:.0f}% cùng hướng | r = {corr:.3f}")

comove_df.round(2)


=== % số sự kiện đi cùng hướng với SPY, và hệ số tương quan ===
AAPL: 74% cùng hướng | r = 0.826
TLT: 52% cùng hướng | r = -0.070
XLF: 84% cùng hướng | r = 0.901


,event_date,direction,SPY_%,AAPL_%,TLT_%,XLF_%
0,2015-12-16,hike,0.97,-3.11,0.32,2.51
1,2016-12-14,hike,0.25,2.22,-0.26,0.64
2,2017-03-15,hike,0.28,1.07,1.19,-0.28
3,2017-06-14,hike,0.17,-0.78,1.40,0.29
4,2017-12-13,hike,-0.24,-0.26,1.14,-0.89
5,2018-03-21,hike,-2.52,-3.68,0.69,-3.49
6,2018-06-13,hike,0.06,-0.22,0.80,-1.53
7,2018-09-26,hike,-0.11,1.88,0.67,-1.94
8,2018-12-19,hike,-3.21,-4.34,1.56,-2.50
9,2019-07-31,cut,-2.20,-0.60,3.06,-3.02


### Trả lời Q2a

- **AAPL và XLF thường đi cùng chiều với thị trường quanh các sự kiện FOMC** (74% và 84%, r = 0.83 và 0.90). Đây là correlation tính trên 31 cửa sổ 3-ngày quanh sự kiện, **không phải beta hay correlation vô điều kiện trên toàn bộ 2013-2026** — kết quả chỉ mô tả đúng bối cảnh đang xét. Kết quả này cũng chưa chứng minh nguyên nhân là beta cao hay thấp, chỉ cho thấy tương quan mạnh. Phần Q2b bên dưới sẽ tách rõ hơn.
- **TLT thì** ít đồng biến với SPY trong các cửa sổ FOMC, phù hợp với đặc điểm phản ứng khác nhau của TLT trước kỳ vọng lãi suất.

## 6. Q2b - Có mã nào phản ứng vượt mức "đi theo thị trường"?

Q2a gộp chung 2 điều: "vốn dĩ đi theo thị trường" và "phản ứng riêng với Fed". **Market Model** tách 2 phần này: ước lượng alpha/beta bình thường của mỗi mã so với SPY (từ cửa sổ trước sự kiện 21 ngày, để tránh lẫn hiệu ứng đón đầu). Sau đó tính **CAR (Cumulative Abnormal Return)** — phần biến động mà beta không giải thích được.

**Cửa sổ ước lượng cho mỗi sự kiện:**

```
|-- 120 ngày ước lượng beta --|-- 21 ngày cách --| [-1, 0, +1]
```

Mỗi sự kiện có cửa sổ ước lượng riêng, vì beta của AAPL hay XLF có thể đổi theo thời gian.

**CAR** ở đây là tổng abnormal return trong cửa sổ sự kiện:

```
CAR[-1,+1] = AR(-1) + AR(0) + AR(+1)
```

**Ghi chú:** cửa sổ `[-1, 0, +1]` được xác định theo `Date` thật của SPY, không theo vị trí trong bảng đã merge — tránh lệch ngày nếu asset thiếu dữ liệu một vài ngày.

In [16]:
def market_model_car(asset_ret_df, ret_col, label, est_len=120, gap=21, window=1):
    merged = pd.merge(
        asset_ret_df,
        spy[["Date", "ret"]].rename(columns={"ret": "mkt_ret"}),
        on="Date",
    ).dropna().reset_index(drop=True)
    dates_list = merged["Date"].tolist()

    def estimate(event_date):
        if event_date not in dates_list:
            return None

        idx = dates_list.index(event_date)
        est_start, est_end = idx - gap - est_len, idx - gap
        if est_start < 0:
            return None

        est_data = merged.iloc[est_start:est_end]

        # Kiểm tra rõ ràng cửa sổ ước lượng có đủ độ dài không, thay vì chỉ
        # dựa vào điều kiện est_start < 0 ở trên (viết explicit dễ đọc hơn).
        if len(est_data) < est_len:
            return None

        X = sm.add_constant(est_data["mkt_ret"])
        model = sm.OLS(est_data[ret_col], X).fit()
        alpha, beta = model.params["const"], model.params["mkt_ret"]

        # Xác định 3 ngày của event window [-1, 0, +1] TRỰC TIẾP theo
        # trading_dates của SPY (nguồn "sự thật" cho ngày sự kiện), rồi mới
        # lọc merged theo đúng các Date đó. Cách này tránh phụ thuộc vào vị
        # trí (positional index) trong merged, vốn có thể lệch khỏi ±1 ngày
        # giao dịch thực tế nếu asset thiếu một vài ngày so với SPY (dù ở
        # bước data-quality-check trước đó, AAPL/TLT/XLF gần như trùng khớp
        # hoàn toàn với ngày của SPY).
        spy_idx = trading_dates.index(event_date)
        lo = max(0, spy_idx - window)
        hi = min(len(trading_dates) - 1, spy_idx + window)
        wanted_dates = set(trading_dates[lo: hi + 1])

        event_rows = merged[merged["Date"].isin(wanted_dates)].sort_values("Date")
        if len(event_rows) == 0:
            return None

        expected = alpha + beta * event_rows["mkt_ret"]
        ar = event_rows[ret_col] - expected

        return {"car": ar.sum(), "beta": beta, "residuals": model.resid}

    results = {}
    for ed in fomc_events["event_trading_day"]:
        r = estimate(ed)
        if r is not None:
            results[ed] = r

    cars = np.array([r["car"] for r in results.values()])
    betas = np.array([r["beta"] for r in results.values()])

    t_stat, p_val = stats.ttest_1samp(cars, 0)
    w_stat, w_p = wilcoxon(cars)
    boot = np.array([RNG.choice(cars, len(cars), replace=True).mean() for _ in range(10_000)])
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    print(f"--- {label} ---")
    print(f"  Số sự kiện: {len(cars)}  Beta trung bình: {betas.mean():.2f}  CAR trung bình: {cars.mean():+.5f}")
    print(f"  t-test: p={p_val:.4f}  Wilcoxon: p={w_p:.4f}  Bootstrap 95% CI: [{ci_low:.5f}, {ci_high:.5f}]")
    print()
    return results


aapl_results = market_model_car(aapl, "aapl_ret", "AAPL")
tlt_results = market_model_car(tlt, "tlt_ret", "TLT")
xlf_results = market_model_car(xlf, "xlf_ret", "XLF")

--- AAPL ---
  Số sự kiện: 31  Beta trung bình: 1.24  CAR trung bình: +0.00072
  t-test: p=0.8450  Wilcoxon: p=0.7498  Bootstrap 95% CI: [-0.00636, 0.00772]

--- TLT ---
  Số sự kiện: 31  Beta trung bình: -0.14  CAR trung bình: +0.00442
  t-test: p=0.1166  Wilcoxon: p=0.1066  Bootstrap 95% CI: [-0.00088, 0.00965]

--- XLF ---
  Số sự kiện: 31  Beta trung bình: 0.98  CAR trung bình: -0.00248
  t-test: p=0.3070  Wilcoxon: p=0.1887  Bootstrap 95% CI: [-0.00694, 0.00224]



## 7. Kiểm tra residual của Market Model

Kiểm tra residual theo từng cửa sổ riêng, không gộp chung tất cả lại. Nếu gộp chung, các năm bình thường sẽ bị trộn với năm biến động mạnh (như 2022), làm kết quả trông "không chuẩn" hơn thực tế.

In [17]:
def diagnostics_summary(results, label):
    rows = []
    for d, r in results.items():
        resid = r["residuals"].values
        if len(resid) < 20:
            continue

        _, jb_p, _, _ = jarque_bera(resid)
        try:
            _, arch_p, _, _ = het_arch(resid)
        except ValueError:
            arch_p = np.nan

        rows.append({"jb_p": jb_p, "arch_p": arch_p, "dw": durbin_watson(resid)})

    diag = pd.DataFrame(rows)
    print(f"{label}: {(diag['jb_p'] < 0.05).mean():.0%} cửa sổ không chuẩn (non-normal) | "
          f"{(diag['arch_p'] < 0.05).mean():.0%} cửa sổ có ARCH | DW trung bình={diag['dw'].mean():.2f}")


diagnostics_summary(aapl_results, "AAPL")
diagnostics_summary(tlt_results, "TLT")
diagnostics_summary(xlf_results, "XLF")


AAPL: 87% cửa sổ không chuẩn (non-normal) | 10% cửa sổ có ARCH | DW trung bình=1.84
TLT: 10% cửa sổ không chuẩn (non-normal) | 10% cửa sổ có ARCH | DW trung bình=2.01
XLF: 52% cửa sổ không chuẩn (non-normal) | 16% cửa sổ có ARCH | DW trung bình=1.97


Kiểm định bổ sung: pooled residual check

In [18]:
# ============================================================
# Pooled ARCH-LM check (so sánh với per-window ở cell trên)
# ============================================================
# Gộp toàn bộ residual của tất cả event window lại thành 1 mảng duy nhất,
# rồi chạy ARCH-LM 1 lần trên mảng gộp đó. Nếu p-value pooled rất nhỏ
# (ví dụ < 0.0001) trong khi % per-window có ARCH chỉ 10-16%, đó là bằng
# chứng cho "pooling artifact": việc gộp residual từ nhiều event window
# (mỗi window ở một market regime khác nhau) làm ARCH-LM trông có vẻ mạnh
# hơn nhiều so với bức tranh thật ở từng window riêng lẻ — không thể quy
# hoàn toàn cho việc cỡ mẫu pooled lớn hơn.

def pooled_arch_check(results, label):
    # Gộp residual của mọi event window thành 1 mảng.
    all_resid = np.concatenate([r["residuals"].values for r in results.values()])

    try:
        _, pooled_p, _, _ = het_arch(all_resid)
    except ValueError:
        pooled_p = np.nan

    print(f"--- {label} ---")
    print(f"  Số quan sát residual gộp: {len(all_resid)}")
    print(f"  Pooled ARCH-LM p-value: {pooled_p:.6f}")
    print()


print("=== So sánh Pooled ARCH-LM vs Per-window (đã tính ở cell trên) ===\n")
pooled_arch_check(aapl_results, "AAPL")
pooled_arch_check(tlt_results, "TLT")
pooled_arch_check(xlf_results, "XLF")

=== So sánh Pooled ARCH-LM vs Per-window (đã tính ở cell trên) ===

--- AAPL ---
  Số quan sát residual gộp: 3720
  Pooled ARCH-LM p-value: 0.000000

--- TLT ---
  Số quan sát residual gộp: 3720
  Pooled ARCH-LM p-value: 0.000000

--- XLF ---
  Số quan sát residual gộp: 3720
  Pooled ARCH-LM p-value: 0.000000



**Lưu ý:** AAPL có 87% cửa sổ bị bác bỏ giả thuyết phân phối chuẩn. Điều này không có nghĩa "mô hình sai" — chỉ có nghĩa residual thường không theo phân phối chuẩn trong các cửa sổ rolling. Đây là lý do dùng thêm Wilcoxon test và bootstrap, không chỉ dựa vào t-test.

**Về pooled ARCH-LM (cell trên):** con số pooled p≈0 chỉ mang tính minh họa cho "pooling artifact" — **không dùng làm bằng chứng chính** về ARCH trong Market Model. Bằng chứng chính vẫn là per-window diagnostics (AAPL 10%, TLT 10%, XLF 16% cửa sổ có ARCH). Pooled ARCH nên được xem là supplementary diagnostic, bị loại khỏi main evidence vì phá vỡ cấu trúc temporal của từng estimation window.

### Q2a — Các mã có đi theo thị trường không?

AAPL và XLF nhìn chung đi cùng chiều với thị trường quanh các sự kiện FOMC. TLT ít đồng biến với SPY trong các cửa sổ FOMC, phù hợp với đặc điểm phản ứng khác nhau của TLT trước kỳ vọng lãi suất.

### Q2b — Có mã nào phản ứng vượt mức đi theo thị trường?

Không mã nào có abnormal return rõ ràng về ý nghĩa thống kê. TLT gần ngưỡng nhất (p ≈ 0.12), nhưng bằng chứng còn yếu.

## 8. Q3 (mở rộng) - VIX có phản ứng đúng như lý thuyết không?

Nếu FOMC giúp giảm bớt sự không chắc chắn, VIX có thể tăng trong giai đoạn trước cuộc họp và giảm trong giai đoạn sau đó. Đây chỉ là một giả thuyết hợp lý, không chắc chắn — vì FOMC cũng có thể làm sự không chắc chắn tăng lên, nếu quyết định bất ngờ hoặc phát biểu của Fed khác kỳ vọng. Phần này chỉ kiểm tra xem mẫu hình đó có xuất hiện trong dữ liệu VIX hay không.

**Cách đo:** `run_up` = thay đổi VIX **từ 3 ngày trước sự kiện đến đúng ngày sự kiện** ($VIX_0 - VIX_{-3}$); `crush` = thay đổi VIX **từ ngày sự kiện đến 3 ngày sau** ($VIX_{+3} - VIX_0$). Đây là 2 phép kiểm định 1-mẫu riêng biệt (mỗi cái kiểm tra $E[\Delta VIX] \neq 0$ trong đúng giai đoạn của nó) — **không phải một test chung** cho toàn bộ pattern "tăng trước + giảm sau".

In [19]:
vix_raw = pd.read_csv("vix_2013_2026.csv", skiprows=[1, 2]).rename(columns={"Price": "Date"})
vix_raw["Date"] = pd.to_datetime(vix_raw["Date"])
vix_lookup = dict(zip(vix_raw["Date"], vix_raw["Close"]))

PRE, POST = 3, 3
vix_rows = []

for ed in event_windows:
    idx = trading_dates.index(ed)
    if idx - PRE < 0 or idx + POST >= len(trading_dates):
        continue

    pre_day, post_day = trading_dates[idx - PRE], trading_dates[idx + POST]
    if ed in vix_lookup and pre_day in vix_lookup and post_day in vix_lookup:
        vix_rows.append({
            "event_date": ed,
            # run_up = VIX[event] - VIX[event-PRE]: thay đổi TRONG giai đoạn
            # từ PRE ngày trước sự kiện đến đúng ngày sự kiện.
            "run_up": vix_lookup[ed] - vix_lookup[pre_day],
            # crush = VIX[event+POST] - VIX[event]: thay đổi TRONG giai đoạn
            # từ ngày sự kiện đến POST ngày sau.
            "crush": vix_lookup[post_day] - vix_lookup[ed],
        })

vix_df = pd.DataFrame(vix_rows)

t_up, p_up = stats.ttest_1samp(vix_df["run_up"], 0)
t_down, p_down = stats.ttest_1samp(vix_df["crush"], 0)

print(f"Số sự kiện được đo: {len(vix_df)}")
print(f"Thay đổi VIX trung bình, từ {PRE} ngày trước ĐẾN ngày sự kiện: {vix_df['run_up'].mean():+.3f}  (t-test p={p_up:.4f})")
print(f"Thay đổi VIX trung bình, từ ngày sự kiện ĐẾN {POST} ngày sau:   {vix_df['crush'].mean():+.3f}  (t-test p={p_down:.4f})")

# In rõ verdict theo ngưỡng p=0.05, để phần diễn giải bên dưới bám sát con số
# thay vì mô tả theo cảm tính.
sig_up = "có ý nghĩa thống kê (p < 0.05)" if p_up < 0.05 else "KHÔNG có ý nghĩa thống kê (p >= 0.05)"
sig_down = "có ý nghĩa thống kê (p < 0.05)" if p_down < 0.05 else "KHÔNG có ý nghĩa thống kê (p >= 0.05)"
print(f"\nGiai đoạn trước sự kiện: {sig_up}")
print(f"Giai đoạn sau sự kiện: {sig_down}")


Số sự kiện được đo: 31
Thay đổi VIX trung bình, từ 3 ngày trước ĐẾN ngày sự kiện: +1.054  (t-test p=0.3671)
Thay đổi VIX trung bình, từ ngày sự kiện ĐẾN 3 ngày sau:   +0.378  (t-test p=0.6369)

Giai đoạn trước sự kiện: KHÔNG có ý nghĩa thống kê (p >= 0.05)
Giai đoạn sau sự kiện: KHÔNG có ý nghĩa thống kê (p >= 0.05)


### Trả lời Q3

VIX tăng nhẹ trước và sau FOMC, nhưng cả hai thay đổi đều không có ý nghĩa thống kê. Vì vậy, chưa có bằng chứng rõ ràng về việc VIX tăng trước FOMC hoặc xảy ra “volatility crush” sau FOMC.